load delta tables

In [0]:
billing = spark.read.format("delta").load("/Workspace/Cross-System Data Monitoring Platform/silver/billing")

analytics = spark.read.format("delta").load("/Workspace/Cross-System Data Monitoring Platform/silver/analytics")

calculating revenue

In [0]:
from pyspark.sql.functions import sum

daily_revenue = billing.groupBy(
    "transaction_date"
).agg(
    sum("amount").alias("billing_revenue")
)

comparsion

In [0]:
comparison = daily_revenue.join(
    analytics,
    daily_revenue.transaction_date == analytics.date
)

drift

In [0]:
from pyspark.sql.functions import abs

drift = comparison.withColumn(
    "difference",
    abs(
        comparison.billing_revenue -
        comparison.total_revenue
    )
)

drift.show()

+----------------+------------------+----------+---------------+-------------+---------------+------------------+
|transaction_date|   billing_revenue|      date|total_customers|total_revenue|avg_transaction|        difference|
+----------------+------------------+----------+---------------+-------------+---------------+------------------+
|      2022-07-31| 7255.839999999999|2022-07-31|             12|      6700.04|         871.24| 555.7999999999993|
|      2023-06-22| 6440.889999999999|2023-06-22|             13|      2781.67|          249.9|3659.2199999999993|
|      2022-03-28|3227.5899999999997|2022-03-28|              8|      1755.72|         271.74|1471.8699999999997|
|      2023-07-15| 4946.719999999999|2023-07-15|              9|      3701.48|         371.13|1245.2399999999993|
|      2023-05-22| 4427.820000000001|2023-05-22|              8|      3295.65|         292.09|1132.1700000000005|
|      2022-12-25|           4738.82|2022-12-25|             14|      3823.94|         3

saving drift reports

In [0]:
drift.write.format("delta").mode("overwrite").save("/Workspace/Cross-System Data Monitoring Platform/gold/drift_report")